<a href="https://colab.research.google.com/github/ABROCK1516/Ukraine-Losses-Weather/blob/main/Ukraine_Losses_Weather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Russian Equipment & Personnel Losses & Front-line Weather**
## *Data Acquisition and Preprocessing – Individual project*
**Alex Brock | C23426432 | TU912/4**

## **Introduction**
This project combines daily figures for Russian equipment and personnel losses in the war in Ukraine with weather observations from near the front line. Weather is often said to shape the fighting, for example mud seasons slowing armoured vehicles, or wind and fog grounding FPV drones which allows for more ground assaults and offensive actions. This project tests whether those patterns can be seen in the data.

I chose this topic because I have been following the war closely since the full scale invasion of Ukraine in 2022. I keep up to date with the war through YouTube channels and the news. From the start of 2023 onwards, I have watched and read about both sides of the war using weather patterns and conditions to their advantage such as fog to aid in assaulting dug-in positions.

I was particularly interested in the military side: how weapons and drones are used, and how much outside conditions like the weather affect them. Commentary on the war often claims that weather changes the fighting, and I wanted to see whether the data actually supports that.

## **Research questions**
The project is built in three layers, each adding a dataset and new questions:

1. **Kaggle data alone:** How have daily losses changed over the course of the war? Has the mix of equipment lost changed, and has the ratio of personnel to vehicles lost changed over time?
2. **Adding weather:** Do daily losses vary with weather conditions such as rain, freezing temperatures, wind and low visibility? Which loss categories are most affected?
3. **Adding combat engagements (extension):** When losses change with the weather, is that because there is less fighting, or because the fighting causes fewer losses?

## **Data sources**

| Source | What it contains | Period | How it was obtained | Role |
|---|---|---|---|---|
| [Kaggle: 2022 Russia Ukraine War (piterfm)](https://www.kaggle.com/datasets/piterfm/2022-ukraine-russian-war) | Daily cumulative Russian equipment losses by category, and personnel losses | 25 Feb 2022 – 10 Oct 2026 (1,689 rows) | Downloaded as two CSV files | Main dataset |
| [pogodaiklimat.ru](http://www.pogodaiklimat.ru/) | 3-hourly weather observations: wind, visibility, cloud cover, weather phenomena, temperature, precipitation | Jan 2023 – Oct 2026 | Web scraping (urllib and BeautifulSoup) | Complementary dataset |
| [ukraine.vefgreining.com](https://ukraine.vefgreining.com/) | Daily number of combat engagements reported by Ukraine's General Staff | 22 Feb 2023 – 16 Oct 2026 | JSON data file | Extension |

Weather is taken from two stations: **occupied Donetsk** (station 34519), which is close to most of the heaviest fighting since 2023, and **Kharkiv** (station 34300), in the Kharkiv region, where fighting continues around Kupiansk, it is used as a second station to check whether results hold.

## **Notes on the data**
- **These are claimed figures.** The loss and engagement figures are reported by Ukraine and have not been independently verified.
- **Personnel figures are estimates.** The dataset itself marks them as approximate (“about” or “more”).
- **"Occupied Donetsk".** The Donetsk weather station is in Russian-occupied territory, and the weather site lists it under Russia.
- **Weather coverage.** The weather site has little data for 2022, so the weather analysis covers 2023 onwards. The Kaggle data is kept in full for Step 1.
- **One weather point for a long front.** The front line is over 1,000 km long, so weather at one station is only an approximation of conditions where the fighting took place.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Import the os library, which lets Python work with folders and file paths
import os
os.chdir("/content/drive/MyDrive/Data ACQ - Project/Data")
!pwd

/content/drive/MyDrive/Data ACQ - Project/Data


## **Step 1: Import the Kaggle dataset**
The dataset is "2022 Russia Ukraine War" by piterfm on Kaggle. It has two CSV files:
running totals of Russian equipment losses and personnel losses.

**About the dataset**
The main dataset is ["2022 Russia Ukraine War" by piterfm](https://www.kaggle.com/datasets/piterfm/2022-ukraine-russian-war) on Kaggle. It records Russian losses in the war as reported by official Ukrainian sources, and has two CSV files:
- `russia_losses_equipment.csv` (1,689 rows × 20 columns): losses by equipment category, such as tanks, armoured personnel carriers (APCs), artillery and drones.
- `russia_losses_personnel.csv` (1,689 rows × 5 columns): personnel losses and prisoners of war (POW).

Both files cover 25 February 2022 to 10 October 2026, with one row per day, and have the same dates, so they can be joined on the `date` column. The figures are **cumulative**: each row gives the total lost since the start of the invasion, not the losses on that day, so daily losses have to be calculated. The data was downloaded on 16 October 2026. Because the dataset is updated weekly, it is not downloaded again, so the results stay reproducible.

In [3]:
# Import pandas, the library used to load and work with tables of data
import pandas as pd

# Load the equipment losses CSV into a DataFrame (a table) called equipment
equipment = pd.read_csv("russia_losses_equipment.csv")

# Load the personnel losses CSV into a DataFrame called personnel
personnel = pd.read_csv("russia_losses_personnel.csv")

In [4]:
# Check the size of the equipment table: (number of rows, number of columns)
equipment.shape

(1689, 20)

In [5]:
# Show the first 10 rows, to see what the data looks like
equipment.head(10)

,date,day,aircraft,helicopter,tank,APC,field artillery,MRL,military auto,fuel tank,drone,naval ship,anti-aircraft warfare,special equipment,mobile SRBM system,greatest losses direction,vehicles and fuel tanks,cruise missiles,submarines,ground robotic systems
0,2026-10-10,1690,443,356,12406,25515,51000,2201,NaN,NaN,556926,35,1699,4812.0,NaN,NaN,160946.0,5126.0,2.0,2917.0
1,2026-10-09,1689,443,356,12402,25509,50950,2199,NaN,NaN,555335,35,1693,4810.0,NaN,NaN,160412.0,5126.0,2.0,2908.0
2,2026-10-08,1688,443,356,12399,25502,50883,2194,NaN,NaN,553578,35,1692,4806.0,NaN,NaN,159795.0,5126.0,2.0,2890.0
3,2026-10-07,1687,443,356,12396,25498,50836,2192,NaN,NaN,551876,35,1690,4800.0,NaN,NaN,159128.0,5126.0,2.0,2878.0
4,2026-10-06,1686,443,356,12396,25494,50785,2184,NaN,NaN,549986,35,1688,4795.0,NaN,NaN,158551.0,5126.0,2.0,2866.0
5,2026-10-05,1685,443,356,12393,25487,50733,2182,NaN,NaN,548095,35,1688,4793.0,NaN,NaN,157985.0,5126.0,2.0,2859.0
6,2026-10-04,1684,443,356,12391,25483,50686,2179,NaN,NaN,546240,35,1686,4788.0,NaN,NaN,157258.0,5126.0,2.0,2837.0
7,2026-10-03,1683,443,356,12391,25477,50637,2178,NaN,NaN,544498,35,1683,4783.0,NaN,NaN,156731.0,5126.0,2.0,2824.0
8,2026-10-02,1682,443,356,12388,25471,50592,2175,NaN,NaN,542853,35,1683,4780.0,NaN,NaN,156309.0,5126.0,2.0,2810.0
9,2026-10-01,1681,443,356,12388,25464,50555,2174,NaN,NaN,541383,35,1683,4774.0,NaN,NaN,155887.0,5126.0,2.0,2803.0


In [6]:
# Show the column names and the data type of each column
equipment.dtypes

,0
date,object
day,int64
aircraft,int64
helicopter,int64
tank,int64
APC,int64
field artillery,int64
MRL,int64
military auto,float64
fuel tank,float64


In [7]:
# Summary statistics for each numeric column:
equipment.describe()

,day,aircraft,helicopter,tank,APC,field artillery,MRL,military auto,fuel tank,drone,naval ship,anti-aircraft warfare,special equipment,mobile SRBM system,vehicles and fuel tanks,cruise missiles,submarines,ground robotic systems
count,1689.000000,1689.000000,1689.000000,1689.000000,1689.000000,1689.000000,1689.000000,65.000000,65.000000,1689.000000,1689.000000,1689.000000,1670.000000,36.000000,1624.000000,1624.000000,1123.000000,161.000000
mean,846.000000,343.914742,303.917703,7302.714624,14825.561279,17909.660746,1046.749556,1047.507692,69.323077,66177.216104,23.708111,788.043813,2282.670060,3.944444,37143.435961,2428.223522,1.266251,2036.956522
std,487.716619,79.985283,60.524649,3836.696267,7956.872953,15393.569483,548.811027,466.162060,7.545917,125776.097509,7.141723,473.888851,1695.212102,0.333333,38493.183090,1501.023468,0.442194,475.056514
min,2.000000,10.000000,7.000000,80.000000,516.000000,49.000000,4.000000,100.000000,60.000000,0.000000,2.000000,0.000000,10.000000,2.000000,1796.000000,84.000000,1.000000,1306.000000
25%,424.000000,308.000000,294.000000,3675.000000,7131.000000,2837.000000,539.000000,600.000000,60.000000,2402.000000,18.000000,289.000000,381.000000,4.000000,6452.750000,1181.250000,1.000000,1636.000000
50%,846.000000,359.000000,326.000000,7974.000000,15307.000000,13959.000000,1104.000000,1178.000000,73.000000,11187.000000,28.000000,853.000000,2425.500000,4.000000,21002.000000,2401.000000,1.000000,1981.000000
75%,1268.000000,421.000000,340.000000,11104.000000,23130.000000,31458.000000,1466.000000,1437.000000,76.000000,51043.000000,28.000000,1207.000000,3942.750000,4.000000,60242.750000,3635.500000,2.000000,2442.000000
max,1690.000000,443.000000,356.000000,12406.000000,25515.000000,51000.000000,2201.000000,1701.000000,76.000000,556926.000000,35.000000,1699.000000,4812.000000,4.000000,160946.000000,5126.000000,2.000000,2917.000000


In [8]:
# Count the missing (empty) values in each column
equipment.isnull().sum()

,0
date,0
day,0
aircraft,0
helicopter,0
tank,0
APC,0
field artillery,0
MRL,0
military auto,1624
fuel tank,1624


In [9]:
# Check the size of the personnel table: (number of rows, number of columns)
personnel.shape

(1689, 5)

In [10]:
# Show the first 5 rows, to see what the data looks like
personnel.head()

,date,day,personnel,personnel*,POW
0,2026-10-10,1690,1550910,about,NaN
1,2026-10-09,1689,1549490,about,NaN
2,2026-10-08,1688,1547780,about,NaN
3,2026-10-07,1687,1545950,about,NaN
4,2026-10-06,1686,1544260,about,NaN


In [11]:
# Show the column names and the data type of each column
personnel.dtypes

,0
date,object
day,int64
personnel,int64
personnel*,object
POW,float64


In [12]:
# Summary statistics for each numeric column:
personnel.describe()

,day,personnel,POW
count,1689.000000,1.689000e+03,62.000000
mean,846.000000,6.250922e+05,386.387097
std,487.716619,4.793099e+05,131.440363
min,2.000000,2.800000e+03,0.000000
25%,424.000000,1.864200e+05,389.000000
50%,846.000000,5.286200e+05,421.000000
75%,1268.000000,1.067100e+06,474.500000
max,1690.000000,1.550910e+06,496.000000


In [13]:
# Count the missing (empty) values in each column
personnel.isnull().sum()

,0
date,0
day,0
personnel,0
personnel*,0
POW,1627


In [14]:
# Compare the date columns row by row; .all() is True only if every row matches
(equipment["date"] == personnel["date"]).all()

np.True_

In [15]:
# Columns with lots of missing values, to check when each one has data
columns_to_check = ["military auto", "fuel tank", "vehicles and fuel tanks",
                    "cruise missiles", "submarines", "ground robotic systems",
                    "mobile SRBM system", "greatest losses direction"]

# Loop through each column name in the list
for col in columns_to_check:
    # Keep only the rows where this column is NOT empty
    filled = equipment[equipment[col].notnull()]
    # Print the column name, first date with data, last date with data, and how many rows have data
    print(col, "|", filled["date"].min(), "to", filled["date"].max(), "|", len(filled), "rows")

military auto | 2022-02-25 to 2022-04-30 | 65 rows
fuel tank | 2022-02-25 to 2022-04-30 | 65 rows
vehicles and fuel tanks | 2022-05-01 to 2026-10-10 | 1624 rows
cruise missiles | 2022-05-01 to 2026-10-10 | 1624 rows
submarines | 2023-09-14 to 2026-10-10 | 1123 rows
ground robotic systems | 2026-05-03 to 2026-10-10 | 161 rows
mobile SRBM system | 2022-03-26 to 2022-04-30 | 36 rows
greatest losses direction | 2022-04-25 to 2023-01-02 | 203 rows


In [16]:
# Keep only the rows where POW has a value
pow_filled = personnel[personnel["POW"].notnull()]
# Print the first date, last date and number of rows with a POW value
print(pow_filled["date"].min(), "to", pow_filled["date"].max(), "|", len(pow_filled), "rows")

2022-02-25 to 2022-04-27 | 62 rows


## **Observaions**
**Structure:** Each row is one day. The numbers are running totals since the start of the invasion, not the losses on that day. The equipment file has 1,689 rows and 20 columns, and the personnel file has 1,689 rows and 5 columns. I checked that both files have the same date in every row (the check returned `True`), so they can be joined on `date`.

**Data types:** The `date` column is stored as `object`, which means pandas sees the dates as text. It needs converting to a real date before it can be used for date calculations. Some columns of whole numbers are stored as floats (decimals). This is because these columns have empty values, and pandas cannot store an empty value in a whole-number column.

**Values:**
- The `day` column starts at 2. The invasion began on 24 Feb 2022 (day 1), so the first row (25 Feb) already includes two days of losses.
- Because the numbers are running totals, the minimum of each column is just its first value. For example, 516 APCs had already been claimed by the first row.
- Drones start at 0. This makes sense, as drones were not widely used early in the war.
- Some new categories start with a high number, such as `cruise missiles` (84) and `ground robotic systems` (1,306). This means earlier losses were added in one go on the first day. Since ground robotic systems were first reported (3 May 2026), about 10 per day have been claimed.

**Missing values:** The empty values are not errors in the data. They happen because the categories in the daily report changed over time:
- **Replaced:** `military auto` and `fuel tank` stop on 30 Apr 2022, and `vehicles and fuel tanks` starts the next day. Together they fill every row (65 + 1,624 = 1,689), so the two old categories were combined into one.
- **Added later:** `cruise missiles` (1 May 2022), `submarines` (14 Sep 2023) and `ground robotic systems` (3 May 2026). Submarines first appear the day after Ukraine's 13 Sep 2023 missile strike on the submarine Rostov-on-Don in Sevastopol, which fits the reports being published the next morning.
- **Dropped:** `mobile SRBM system` was only reported from 26 Mar to 30 Apr 2022. `greatest losses direction` contains place names, not numbers, and was only reported from 25 Apr 2022 to 2 Jan 2023, with gaps, so it is not used.
- **Personnel:** `POW` was only reported from 25 Feb to 27 Apr 2022. After that it stopped being published, possibly because Ukraine chose not to share prisoner numbers.

This means a category can only be compared over time for the period it was reported.



## **Preparing the data: From running totals to daily losses**
The figures are running totals, but the research questions are about losses on each day. To get daily losses, I:
1. convert `date` from text to real dates, so pandas can sort and compare them
2. join the equipment and personnel files into one table on `date`
3. sort the rows from oldest to newest (the files are newest first)
4. subtract each day's total from the next day's total

## **Converting the dates**
`pd.to_datetime()` turns text such as "2026-10-10" into a real date. Before this, pandas treats the dates as plain text (`object`).

In [17]:
equipment["date"] = pd.to_datetime(equipment["date"])  # convert equipment dates from text to real dates
personnel["date"] = pd.to_datetime(personnel["date"])  # same for personnel

print(equipment["date"].dtype)  # should now show datetime64 instead of object
print(personnel["date"].dtype)  # same check for personnel

datetime64[ns]
datetime64[ns]


### **Joining the two files**
Both files have the same 1,689 dates (checked above), so I join them on `date` with `pd.merge`. I leave out the personnel file's `day` column, because the equipment file already has it and keeping both would create two copies.

In [18]:
personnel_cols = personnel[["date", "personnel", "personnel*", "POW"]]  # keep only the columns needed (double brackets = a list of columns)

losses = pd.merge(equipment, personnel_cols, on="date", how="outer")  # join on date; "outer" keeps dates found in either file, so mismatches would show as empty values

losses.shape  # expect 1,689 rows and 23 columns (20 from equipment + 3 from personnel)

(1689, 23)

### **Sorting by date**
`.sort_values("date")` puts the rows in order from oldest to newest. `.reset_index(drop=True)` renumbers the rows 0, 1, 2... in the new order. I then check there are no missing days, by measuring the gap between each date and the one before it.

In [19]:
losses = losses.sort_values("date")  # oldest first, so each row follows the day before it
losses = losses.reset_index(drop=True)  # renumber the rows; drop=True throws away the old row numbers

losses.head()  # the first row should now be 25 Feb 2022

,date,day,aircraft,helicopter,tank,APC,field artillery,MRL,military auto,fuel tank,...,special equipment,mobile SRBM system,greatest losses direction,vehicles and fuel tanks,cruise missiles,submarines,ground robotic systems,personnel,personnel*,POW
0,2022-02-25,2,10,7,80,516,49,4,100.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2800,about,0.0
1,2022-02-26,3,27,26,146,706,49,4,130.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4300,about,0.0
2,2022-02-27,4,27,26,150,706,50,4,130.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4500,about,0.0
3,2022-02-28,5,29,29,150,816,74,21,291.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5300,about,0.0
4,2022-03-01,6,29,29,198,846,77,24,305.0,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5710,about,200.0


In [20]:
losses["date"].diff().value_counts()  # gap between each date and the one before; every gap should be 1 day

,count
date,
1 days,1688


### **Calculating daily losses**
`.diff()` subtracts the previous row's value from each row. For a running total, that gives the losses claimed on that day. The first row has no previous day, so its daily value is empty (`NaN`).

I calculate daily losses for every category reported after spring 2022. `military auto`, `fuel tank` and `mobile SRBM system` are left out, because they were only reported for the first two months of the war.

In [21]:
# The running-total columns to turn into daily losses
loss_columns = ["aircraft", "helicopter", "tank", "APC", "field artillery", "MRL",
                "drone", "naval ship", "anti-aircraft warfare", "special equipment",
                "vehicles and fuel tanks", "cruise missiles", "submarines",
                "ground robotic systems", "personnel"]

for col in loss_columns:  # go through each column in the list
    losses["daily " + col] = losses[col].diff()  # today's total minus yesterday's = losses claimed that day

losses[["date", "tank", "daily tank", "personnel", "daily personnel"]].head(10)  # check a few columns side by side

,date,tank,daily tank,personnel,daily personnel
0,2022-02-25,80,NaN,2800,NaN
1,2022-02-26,146,66.0,4300,1500.0
2,2022-02-27,150,4.0,4500,200.0
3,2022-02-28,150,0.0,5300,800.0
4,2022-03-01,198,48.0,5710,410.0
5,2022-03-02,211,13.0,5840,130.0
6,2022-03-03,217,6.0,9000,3160.0
7,2022-03-04,251,34.0,9166,166.0
8,2022-03-05,269,18.0,10000,834.0
9,2022-03-06,285,16.0,11000,1000.0


### **Checking the daily figures**
A running total should never go down, so a negative daily figure would mean the source corrected an earlier number. I count the negative days in each column, then look at summary statistics for the main categories.

In [22]:
for col in loss_columns:  # go through each loss category
    negatives = (losses["daily " + col] < 0).sum()  # count the days where the daily figure is below zero
    print(col, negatives)  # print the category and its count

aircraft 2
helicopter 0
tank 1
APC 1
field artillery 1
MRL 2
drone 0
naval ship 0
anti-aircraft warfare 2
special equipment 2
vehicles and fuel tanks 0
cruise missiles 2
submarines 0
ground robotic systems 0
personnel 0


In [23]:
# Summary statistics for the daily figures of the main categories
losses[["daily tank", "daily APC", "daily field artillery", "daily drone", "daily personnel"]].describe()

,daily tank,daily APC,daily field artillery,daily drone,daily personnel
count,1688.000000,1688.000000,1688.000000,1688.000000,1688.000000
mean,7.302133,14.809834,30.184242,329.932464,917.126777
std,6.535839,13.564484,22.665864,626.922771,420.982929
min,-2.000000,-14.000000,-10.000000,0.000000,0.000000
25%,3.000000,5.000000,10.000000,8.000000,600.000000
50%,5.000000,11.000000,26.000000,36.000000,970.000000
75%,10.000000,20.000000,46.000000,205.250000,1210.000000
max,66.000000,190.000000,122.000000,2719.000000,3160.000000


In [24]:
for col in ["tank", "APC", "field artillery"]:  # the columns with negative minimums
    neg_days = losses[losses["daily " + col] < 0]  # keep only the rows where the daily figure is negative
    print(neg_days[["date", "daily " + col]])  # show the date and the negative value

zero_personnel = losses[losses["daily personnel"] == 0]  # rows where no personnel losses were added
print(zero_personnel[["date", "personnel", "daily personnel"]])  # show when they happened

          date  daily tank
457 2023-05-28        -2.0
          date  daily APC
230 2022-10-13      -14.0
           date  daily field artillery
1468 2026-03-04                  -10.0
         date  personnel  daily personnel
10 2022-03-07      11000              0.0
12 2022-03-09      12000              0.0
13 2022-03-10      12000              0.0
14 2022-03-11      12000              0.0
15 2022-03-12      12000              0.0
16 2022-03-13      12000              0.0
17 2022-03-14      12000              0.0


In [25]:
losses[["date", "tank", "daily tank"]][454:461]  # rows 454 to 460: three days either side of the tank correction

,date,tank,daily tank
454,2023-05-25,3795,3.0
455,2023-05-26,3796,1.0
456,2023-05-27,3799,3.0
457,2023-05-28,3797,-2.0
458,2023-05-29,3801,4.0
459,2023-05-30,3802,1.0
460,2023-05-31,3802,0.0


## **Observations on daily losses**

**Dates:** After sorting, every gap between consecutive dates is exactly one day (1,688 gaps for 1,689 rows), so there are no missing or repeated days.

**Negative days:** A running total should never go down, but 13 days across 8 columns do. Aircraft, MRL, anti-aircraft warfare, special equipment and cruise missiles have two negative days each, and tank, APC and field artillery have one each:
- APC: −14 on 13 Oct 2022
- tank: −2 on 28 May 2023
- field artillery: −10 on 4 Mar 2026

These look like corrections, where the source lowered a total it had reported earlier. For the tank correction, the days either side are normal (+3 the day before, +4 the day after), so it isn't undoing one unusually high day. Drones, vehicles and personnel have no negative days. The negative days are small and rare, so I note them as a data issue to deal with in cleaning.

**Days with no personnel losses:** Seven days in early March 2022 (7 Mar and 9–14 Mar) show zero daily personnel losses. The totals at that time are round figures (11,000 and 12,000), which suggests the early estimates were rounded and only updated every few days. So daily personnel figures for early 2022 show when the estimate was updated, not when losses happened. This doesn't affect the weather analysis, which starts in January 2023.

**Drones vs tanks:** The median daily drone loss (36) is much higher than for tanks (5), which fits drones being used in far greater numbers and being cheaper and more expendable. The mean for drones (330) is about nine times the median, because a small number of very large days pull it up. Drone use grew sharply over the war, so a whole-war median mixes very different phases. In Step 4 I compare the years separately and use a log scale or rolling average for drones.

## **Step 2: Scraping the weather data**

The complementary dataset is weather observations scraped from [pogodaiklimat.ru](http://www.pogodaiklimat.ru), a Russian weather archive. It publishes readings from weather stations every 3 hours (8 per day), including wind speed, visibility, cloud cover, weather phenomena such as fog and rain, temperature and precipitation. Visibility and weather phenomena are needed to test whether fog and wind affect drone losses, and most weather sites only give daily summaries without them.

I scrape two stations: **occupied Donetsk** (station 34519) and **Kharkiv** (station 34300). The site has one page per station per month, and the URL sets which station and month is shown, for example:

`http://www.pogodaiklimat.ru/weather.php?id=34519&bday=1&fday=31&amonth=3&ayear=2024&bot=2`

- `id`: the station number
- `bday` and `fday`: the first and last day of the month
- `amonth` and `ayear`: the month and year
- `bot`: see the ethics section below

**The steps in this section are:**
1. Check the site's robots.txt and terms of use.
2. Fetch one test page and inspect how it is built.
3. Write a parser that turns one page into a table.
4. Scrape every month for both stations, saving the raw pages to Drive first.
5. Check how many days each month has data for, then combine the 3-hourly readings into one row per day.

## **Checking robots.txt**
Before scraping, I check the site's robots.txt file, which tells automated programs which parts of the site they shouldn't access (week 2 ethics). I fetch it with the same urllib method I'll use for the weather pages.

In [27]:
import urllib.request  # module from week 2 for fetching web pages

robots_url = "http://www.pogodaiklimat.ru/robots.txt"  # the site's rules for automated programs

request = urllib.request.Request(robots_url)  # build the request
request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request politely
opener = urllib.request.build_opener()  # object that sends requests
response = opener.open(request)  # send the request
robots_text = response.read().decode("utf-8")  # read the bytes and turn them into text

print(robots_text)  # display the rules

User-agent: *
Disallow: /admin/
Disallow: /metar/
Disallow: /photo/
Disallow: /synop/
Disallow: /summary/
Disallow: /usasummary/
Disallow: /extremal.php?
Disallow: /extremal1.php?
Disallow: /dextremal.php?
Disallow: /dextremal1.php?

User-agent: Yandex
Disallow: /admin/
Disallow: /metar/
Disallow: /photo/
Disallow: /synop/
Disallow: /summary/
Disallow: /usasummary/
Disallow: /extremal.php?
Disallow: /extremal1.php?
Disallow: /dextremal.php?
Disallow: /dextremal1.php?
Host: www.pogodaiklimat.ru


In [28]:
from bs4 import BeautifulSoup

agreement_url = "http://www.pogodaiklimat.ru/agreement.php"  # the site's user agreement page

request = urllib.request.Request(agreement_url)  # build the request
request.add_header("User-Agent", "Mozilla/5.0 (student project, TU Dublin)")  # identify the request
response = opener.open(request)  # send it, reusing the opener from the robots.txt cell
agreement_html = response.read().decode("utf-8")  # read the bytes and turn them into text

agreement_soup = BeautifulSoup(agreement_html, "html.parser")  # turn the HTML into a searchable object
agreement_text = agreement_soup.get_text()  # keep only the text, without the HTML tags

start_3_1 = agreement_text.find("3.1 ")  # position where clause 3.1 starts (the space stops it matching 3.10)
end_3_1 = agreement_text.find("3.2 ")  # position where clause 3.2 starts, i.e. where 3.1 ends
print(agreement_text[start_3_1:end_3_1])  # print clause 3.1 only

start_3_5 = agreement_text.find("3.5 ")  # position where clause 3.5 starts
end_3_7 = agreement_text.find("3.8 ")  # position where clause 3.8 starts, i.e. where 3.7 ends
print(agreement_text[start_3_5:end_3_7])  # print clauses 3.5, 3.6 and 3.7

3.1 Сайт предназначен для использования исключительно в личных некоммерческих целях. Пользователь обязан пользоваться Сайтом в порядке, определенном Соглашением.

3.5 Полное или частичное копирование, воспроизведение, распространение, размещение в свободном доступе в сети Интернет текстовых и графических материалов Сайта, в том числе - элементов дизайна, любое их использование в средствах массовой информации и/или в коммерческих целях без предварительного письменного разрешения Администрации Сайта запрещено.
3.6 Запрещено копирование, распространение, размещение в свободном доступе в сети Интернет Данных, находящихся в разделе Саммари, недоступном для Пользователей без предварительной авторизации.
3.7 При размещении в свободном доступе Данных с Сайта в сети Интернет обязательно необходима активная ссылка на источник информации (Источник: сайт «Погода и Климат»).



## **Terms of use**
**3.1** The site is intended for use solely for personal, non-commercial purposes. Users must use the site in accordance with the rules set out in the Agreement.

**3.5** Full or partial copying, reproduction, distribution, posting in public access on the Internet of the site's text and graphic materials, including design elements, as well as any use in the media and/or for commercial purposes without prior written permission from the Site Administration is prohibited.

**3.6** Copying, distributing, or posting in public access on the Internet data from the Summary section, which is not available to users without prior authorization, is prohibited.

**3.7** When posting data from the site in public access on the Internet, an active link to the information source (Source: "Weather and Climate" website) is required.

*(checked with google translate)*

## **Scraping Ethics**

**Robots.txt:** I checked the site's robots.txt file before scraping. The rules for all user agents (`User-agent: *`) apply to my scraper. `/weather.php`, the only page I request, is not disallowed, and the file sets no crawl delay. I don't access any of the disallowed paths, such as `/summary/`.

**Terms of Use:** Printed in the terms-of-use check above
- The site is for personal, non-commercial use only **(clause 3.1).** This project is non-commercial coursework.
- The site's own text, graphics and design may not be copied or republished without permission **(clause 3.5)**. So the raw HTML pages stay private in my Google Drive, and only the extracted weather data goes on GitHub.
- Data from the members-only Summary section may not be shared **(clause 3.6)**. I don't use it.
- When the site's data is published online, a link to the source is required **(clause 3.7)**. The data is credited below and in the README.


**Limiting Requests:** I will avoid overloading the site and add a delay (2-3 seconds) between requests to not put a burst of load on the site's server.

**Necessary data only:** Only the two stations and months necessary for my analysis. The pages contain public weather observations only. No login is used and no personal data is collected

**Identifying the Scraper:** Each request sends a User-Agent header that identifies it as a student project, rather than hiding what it is.

**Personal data:** The pages contain public weather observations only. No login is used and no personal data is collected.

**The `bot` parameter:** *To be confirmed: which answer the `bot` value in the URL sends to the site's "robot / alien / human" question.*

**Source:** Weather data from [Pogoda i Klimat](http://www.pogodaiklimat.ru) (Источник: сайт «Погода и Климат»).